In [1]:
# Welcome to your new notebook
# Type here in the cell editor to add code!
from pyspark.sql.functions import *
from pyspark.sql.window import Window

customers   = spark.read.option("header", True).csv("Files/Bronze/customers.csv")
orders      = spark.read.option("header", True).csv("Files/Bronze/orders.csv")
items       = spark.read.option("header", True).csv("Files/Bronze/order_items.csv")
products    = spark.read.option("header", True).csv("Files/Bronze/products.csv")
payments    = spark.read.option("header", True).csv("Files/Bronze/order_payments_dataset.csv")
reviews     = spark.read.option("header", True).csv("Files/Bronze/order_reviews.csv")
sellers     = spark.read.option("header", True).csv("Files/Bronze/sellers.csv")
geo         = spark.read.option("header", True).csv("Files/Bronze/geolocation.csv")
translation = spark.read.option("header", True).csv("Files/Bronze/product_category_name_translation.csv")

StatementMeta(, 0cf6f237-e4ce-4093-854b-490df7ce8c6d, 3, Finished, Available, Finished, False)

In [2]:
customers = (customers
    .withColumn("customer_zip_code_prefix", col("customer_zip_code_prefix").cast("integer"))
    .withColumn("customer_city", initcap(trim(col("customer_city"))))
    .withColumn("customer_state", upper(trim(col("customer_state"))))
    .dropDuplicates(["customer_id"]))

customers.show(5)

StatementMeta(, 0cf6f237-e4ce-4093-854b-490df7ce8c6d, 5, Finished, Available, Finished, False)

+--------------------+--------------------+------------------------+-------------------+--------------+
|         customer_id|  customer_unique_id|customer_zip_code_prefix|      customer_city|customer_state|
+--------------------+--------------------+------------------------+-------------------+--------------+
|00050bf6e01e69d5c...|e3cf594a99e810f58...|                   98700|               Ijui|            RS|
|000598caf2ef41174...|7e0516b486e92ed3f...|                   35540|           Oliveira|            MG|
|0013cd8e350a7cc76...|334fed5abcee3aa96...|                    3585|          Sao Paulo|            SP|
|0015bc9fd2d539544...|490c854539b21598c...|                   12233|Sao Jose Dos Campos|            SP|
|001df1ee5c36767aa...|46b44ab325f78e5bb...|                    1030|          Sao Paulo|            SP|
+--------------------+--------------------+------------------------+-------------------+--------------+
only showing top 5 rows



In [3]:
date_cols = ["order_purchase_timestamp", "order_approved_at",
             "order_delivered_carrier_date", "order_delivered_customer_date",
             "order_estimated_delivery_date"]

for c in date_cols:
    orders = orders.withColumn(c, to_timestamp(col(c), "M/d/yyyy H:mm"))

orders = (orders
    .withColumn("OrderYear", year("order_purchase_timestamp"))
    .withColumn("OrderMonth", month("order_purchase_timestamp"))
    .withColumn("OrderDay", dayofmonth("order_purchase_timestamp"))
    .withColumn("MonthName", date_format("order_purchase_timestamp", "MMMM"))
    .withColumn("Quarter", quarter("order_purchase_timestamp"))
    .withColumn("WeekDay", date_format("order_purchase_timestamp", "EEEE"))
    .withColumn("DeliveryDays", datediff(col("order_delivered_customer_date"), col("order_purchase_timestamp")))
    .withColumn("LateDeliveryDays", datediff(col("order_delivered_customer_date"), col("order_estimated_delivery_date")))
    .withColumn("DeliveryStatus",
        when(col("order_delivered_customer_date").isNull(), "Not Delivered Yet")
        .when(col("LateDeliveryDays") > 0, "Late")
        .otherwise("On Time"))
    .dropDuplicates(["order_id"]))

orders.show(5)

StatementMeta(, 0cf6f237-e4ce-4093-854b-490df7ce8c6d, 8, Finished, Available, Finished, False)

+--------------------+--------------------+------------+------------------------+-------------------+----------------------------+-----------------------------+-----------------------------+---------+----------+--------+---------+-------+---------+------------+----------------+--------------+
|            order_id|         customer_id|order_status|order_purchase_timestamp|  order_approved_at|order_delivered_carrier_date|order_delivered_customer_date|order_estimated_delivery_date|OrderYear|OrderMonth|OrderDay|MonthName|Quarter|  WeekDay|DeliveryDays|LateDeliveryDays|DeliveryStatus|
+--------------------+--------------------+------------+------------------------+-------------------+----------------------------+-----------------------------+-----------------------------+---------+----------+--------+---------+-------+---------+------------+----------------+--------------+
|00018f77f2f0320c5...|f6dd3ec061db4e398...|   delivered|     2017-04-26 10:53:00|2017-04-26 11:05:00|         2017-05-

In [4]:
items = (items
    .withColumn("price", col("price").cast("double"))
    .withColumn("freight_value", col("freight_value").cast("double"))
    .withColumn("order_item_id", col("order_item_id").cast("integer"))
    .withColumn("TotalValue", col("price") + col("freight_value"))
    .dropDuplicates(["order_id", "order_item_id"]))

items.show(5)

StatementMeta(, 0cf6f237-e4ce-4093-854b-490df7ce8c6d, 11, Finished, Available, Finished, False)

+--------------------+-------------+--------------------+--------------------+-------------------+-----+-------------+------------------+
|            order_id|order_item_id|          product_id|           seller_id|shipping_limit_date|price|freight_value|        TotalValue|
+--------------------+-------------+--------------------+--------------------+-------------------+-----+-------------+------------------+
|00018f77f2f0320c5...|            1|e5f2d52b802189ee6...|dd7ddc04e1b6c2c61...|     5/3/2017 11:05|239.9|        19.93|            259.83|
|00061f2a7bc09da83...|            1|d63c1011f49d98b97...|cc419e0650a3c5ba7...|    3/29/2018 22:28|59.99|         8.88|             68.87|
|0014ae671de39511f...|            1|23365beed316535b4...|92eb0f42c21942b65...|     5/29/2017 3:15| 16.5|         14.1|              30.6|
|0015ebb40fb17286b...|            1|50fd2b788dc166edd...|8b321bb669392f516...|     1/18/2018 9:11| 21.9|         15.1|              37.0|
|001ab0a7578dd66cd...|            

In [5]:
products = (products
    .fillna({"product_category_name": "unknown"})
    .fillna(0, subset=["product_name_lenght", "product_description_lenght", "product_photos_qty",
                        "product_weight_g", "product_length_cm", "product_height_cm", "product_width_cm"])
    .join(translation, "product_category_name", "left")
    .withColumn("product_category_name_english", coalesce(col("product_category_name_english"), lit("Unknown")))
    .withColumn("product_weight_g", col("product_weight_g").cast("double"))
    .dropDuplicates(["product_id"]))

products.show(5)

StatementMeta(, 0cf6f237-e4ce-4093-854b-490df7ce8c6d, 14, Finished, Available, Finished, False)

+---------------------+--------------------+-------------------+--------------------------+------------------+----------------+-----------------+-----------------+----------------+-----------------------------+
|product_category_name|          product_id|product_name_lenght|product_description_lenght|product_photos_qty|product_weight_g|product_length_cm|product_height_cm|product_width_cm|product_category_name_english|
+---------------------+--------------------+-------------------+--------------------------+------------------+----------------+-----------------+-----------------+----------------+-----------------------------+
|           perfumaria|00066f42aeeb9f300...|                 53|                       596|                 6|           300.0|               20|               16|              16|                    perfumery|
|           automotivo|00088930e925c41fd...|                 56|                       752|                 4|          1225.0|               55|           

In [6]:
payments = (payments
    .withColumn("payment_sequential", col("payment_sequential").cast("integer"))
    .withColumn("payment_installments", col("payment_installments").cast("integer"))
    .withColumn("payment_value", col("payment_value").cast("double"))
    .withColumn("InstallmentBucket",
        when(col("payment_installments") <= 1, "Single Payment")
        .when(col("payment_installments") <= 6, "2-6 Installments")
        .otherwise("7+ Installments")))

payments.show(5)

StatementMeta(, 0cf6f237-e4ce-4093-854b-490df7ce8c6d, 17, Finished, Available, Finished, False)

+--------------------+------------------+------------+--------------------+-------------+-----------------+
|            order_id|payment_sequential|payment_type|payment_installments|payment_value|InstallmentBucket|
+--------------------+------------------+------------+--------------------+-------------+-----------------+
|b81ef226f3fe1789b...|                 1| credit_card|                   8|        99.33|  7+ Installments|
|a9810da82917af2d9...|                 1| credit_card|                   1|        24.39|   Single Payment|
|25e8ea4e93396b6fa...|                 1| credit_card|                   1|        65.71|   Single Payment|
|ba78997921bbcdc13...|                 1| credit_card|                   8|       107.78|  7+ Installments|
|42fdf880ba16b47b5...|                 1| credit_card|                   2|       128.45| 2-6 Installments|
+--------------------+------------------+------------+--------------------+-------------+-----------------+
only showing top 5 rows



In [7]:
reviews = (reviews
    .withColumn("review_score", col("review_score").cast("integer"))
    .fillna({"review_comment_title": "No Title", "review_comment_message": "No Comment"})
    .withColumn("HasWrittenReview", when(col("review_comment_message") != "No Comment", True).otherwise(False))
    .dropDuplicates(["review_id"]))

reviews.show(5)

StatementMeta(, 0cf6f237-e4ce-4093-854b-490df7ce8c6d, 20, Finished, Available, Finished, False)

+--------------------+---------------+------------+--------------------+----------------------+--------------------+-----------------------+----------------+
|           review_id|       order_id|review_score|review_comment_title|review_comment_message|review_creation_date|review_answer_timestamp|HasWrittenReview|
+--------------------+---------------+------------+--------------------+----------------------+--------------------+-----------------------+----------------+
| Ao se colocar o ...|       na CAPA |        NULL|        o Microfone |   ficam Bloquados ...|       4/4/2018 0:00|         4/6/2018 16:36|            true|
| Gostaria que a e...| 9/24/2017 0:00|        NULL|            No Title|            No Comment|                NULL|                   NULL|           false|
| Mas chegou dentr...| 8/10/2018 0:00|        NULL|            No Title|            No Comment|                NULL|                   NULL|           false|
|           Obrigada"| 3/29/2018 0:00|        NULL| 

In [8]:
sellers = (sellers
    .withColumn("seller_zip_code_prefix", col("seller_zip_code_prefix").cast("integer"))
    .withColumn("seller_city", initcap(trim(col("seller_city"))))
    .withColumn("seller_state", upper(trim(col("seller_state"))))
    .dropDuplicates(["seller_id"]))

sellers.show(5)

StatementMeta(, 0cf6f237-e4ce-4093-854b-490df7ce8c6d, 23, Finished, Available, Finished, False)

+--------------------+----------------------+-----------+------------+
|           seller_id|seller_zip_code_prefix|seller_city|seller_state|
+--------------------+----------------------+-----------+------------+
|0015a82c2db000af6...|                  9080|Santo Andre|          SP|
|001cca7ae9ae17fb1...|                 29156|  Cariacica|          ES|
|001e6ad469a905060...|                 24754|Sao Goncalo|          RJ|
|002100f778ceb8431...|                 14405|     Franca|          SP|
|003554e2dce176b55...|                 74565|    Goiania|          GO|
+--------------------+----------------------+-----------+------------+
only showing top 5 rows



In [9]:
geo_clean = (geo
    .withColumn("geolocation_zip_code_prefix", col("geolocation_zip_code_prefix").cast("integer"))
    .withColumn("geolocation_lat", col("geolocation_lat").cast("double"))
    .withColumn("geolocation_lng", col("geolocation_lng").cast("double"))
    .dropDuplicates())

geo_agg = (geo_clean.groupBy("geolocation_zip_code_prefix")
    .agg(avg("geolocation_lat").alias("lat"),
         avg("geolocation_lng").alias("lng"),
         first("geolocation_city").alias("city"),
         first("geolocation_state").alias("state")))

print("before:", geo.count(), "after dedup+agg:", geo_agg.count())
geo_agg.show(5)

StatementMeta(, 0cf6f237-e4ce-4093-854b-490df7ce8c6d, 26, Finished, Available, Finished, False)

before: 1000163 after dedup+agg: 19015
+---------------------------+-------------------+-------------------+---------+-----+
|geolocation_zip_code_prefix|                lat|                lng|     city|state|
+---------------------------+-------------------+-------------------+---------+-----+
|                       1005|-23.549547473076927| -46.63640613076923|sao paulo|   SP|
|                       1016|-23.549094880000002| -46.63211163857142|sao paulo|   SP|
|                       1019|-23.551762192941176| -46.63085454235294|são paulo|   SP|
|                       1025|-23.540303051666665|-46.631086924166674|sao paulo|   SP|
|                       1030|-23.540250349285714|-46.633066558571436|sao paulo|   SP|
+---------------------------+-------------------+-------------------+---------+-----+
only showing top 5 rows



In [10]:
silver_tables = {
    "silver_customers": customers,
    "silver_orders": orders,
    "silver_order_items": items,
    "silver_products": products,
    "silver_payments": payments,
    "silver_reviews": reviews,
    "silver_sellers": sellers,
    "silver_geolocation": geo_agg
}

for name, df in silver_tables.items():
    df.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(name)
    print(f"saved: {name}")

StatementMeta(, 0cf6f237-e4ce-4093-854b-490df7ce8c6d, 28, Finished, Available, Finished, False)

saved: silver_customers
saved: silver_orders
saved: silver_order_items
saved: silver_products
saved: silver_payments
saved: silver_reviews
saved: silver_sellers
saved: silver_geolocation
